<a href="https://colab.research.google.com/github/Vendetta-ops-cy/English-Igbo-Hate-Speech-Detection/blob/main/hate_speech_model_final2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import transformers
import datasets
import torch
import pandas as pd
import sklearn

print("All packages imported successfully!")

Run Imports

In [ ]:
import torch

import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available()else "cpu")

Load Dataset and Preprocessing

In [ ]:
import pandas as pd
import re

# insert your pdf here
df = pd.read_csv("your_dataset.csv")

print(df.head())

def clean_text(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["text"].apply(clean_text)

print(df[["text","clean_text"]].head())

Overlap Detection

In [ ]:
# Load sentence embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Convert texts into embeddings
embeddings = model.encode(
    df["text"].tolist(),
    show_progress_bar=True
)

# Calculate similarity
similarity = cosine_similarity(embeddings)

# Similarity threshold
THRESHOLD = 0.95

remove = set()

for i in range(len(df)):
    if i in remove:
        continue

    for j in range(i + 1, len(df)):
        if j in remove:
            continue

        if similarity[i][j] >= THRESHOLD:
            remove.add(j)

# Create cleaned dataset
cleaned_df = df.drop(index=list(remove)).reset_index(drop=True)

cleaned_df.to_csv("cleaned_dataset.csv", index=False)

print("Original dataset:", len(df))
print("Removed overlapping texts:", len(remove))
print("Cleaned dataset:", len(cleaned_df))
df = pd.read_csv("cleaned_dataset.csv")

Split[train, test, val]

In [ ]:
# from datasets import dataset
train_df = df[df["split"]=="train"]
val_df = df[df["split"]=="val"]
test_df = df[df["split"]=="test"]

label_map = {
 "normal":0,
 "offensive":1,
 "hate_speech":2,
 "counter_speech":3
}

train_df["label"]=train_df["label"].map(label_map)
val_df["label"]=val_df["label"].map(label_map)
test_df["label"]=test_df["label"].map(label_map)

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

Tokenization

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length= 128)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=4
)

model.to(device)

Create Model(TrainingArguments)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    # logging_dir="./logs",
    save_strategy="steps",
    save_steps=500,
    logging_steps=100,
    fp16=True,
    # downloader_pin_memory=False
)

import os
os.environ["TENSORBOARD_LOGGIN_DIR"] = "./logs"
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

predictions = trainer.predict(test_dataset)

print(predictions.metrics)

Trainer[trainer.train()]

In [ ]:
trainer.train()

# generate predictions FIRST
predictions = trainer.predict(test_dataset)

y_true = predictions.label_ids
y_pred = predictions.predictions.argmax(-1)

cm = confusion_matrix(y_true, y_pred)

sns.heatmap(cm, annot=True, fmt="d")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

trainer.save_model("hate_speech_model")
tokenizer.save_pretrained("hate_speech_model")

def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True,padding=True)

    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    prediction = torch.argmax(outputs.logits, dim=1).item()

    label_names = ["normal","offensive","hate_speech","counter_speech"]

    return label_names[prediction]

In [ ]:
predict("you people are stupid,unu bu ndi ara")

In [ ]:
set(train_df["text"]).intersection(set(test_df["text"]))

In [ ]:
train_df['label'].value_counts()